In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ==============================
# 1. ĐỌC FILE EXCEL
# ==============================

file_path = "K58KTP.xlsx"

raw_df = pd.read_excel(file_path)

# ==============================
# 2. LẤY MSSV + TÊN
# ==============================

student_ids = raw_df.iloc[0, 3:].values
student_names = raw_df.iloc[1, 3:].values

# ==============================
# 3. LẤY DỮ LIỆU ĐIỂM
# ==============================

score_data = raw_df.iloc[3:, 3:]

subjects = raw_df.iloc[3:, 1].values

score_data.index = subjects

# chuyển sang số
score_data = score_data.apply(pd.to_numeric, errors='coerce')

# ==============================
# 4. CHUYỂN MA TRẬN
# ==============================

student_scores = score_data.T

student_scores.columns = subjects

# ==============================
# 5. XỬ LÝ GIÁ TRỊ THIẾU
# ==============================

# Điền NaN bằng trung bình từng môn
imputer = SimpleImputer(strategy='mean')

X_imputed = imputer.fit_transform(student_scores)

# ==============================
# 6. CHUẨN HÓA DỮ LIỆU
# ==============================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed)

# ==============================
# 7. PHÂN CỤM KMEANS
# ==============================

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

# ==============================
# 8. KẾT QUẢ PHÂN CỤM
# ==============================

result_df = pd.DataFrame({
    'MSSV': student_ids,
    'Tên sinh viên': student_names,
    'Cụm': clusters
})

print(result_df)

# ==============================
# 9. PCA GIẢM CHIỀU
# ==============================

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    'PCA1': X_pca[:, 0],
    'PCA2': X_pca[:, 1],
    'Cluster': clusters
})

# ==============================
# 10. VẼ BIỂU ĐỒ
# ==============================

plt.figure(figsize=(10, 6), dpi=90)

sns.scatterplot(
    data=plot_df,
    x='PCA1',
    y='PCA2',
    hue='Cluster',
    palette='Set1',
    s=120
)

plt.title('Phân cụm sinh viên bằng KMeans')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')

plt.grid(True)

plt.show()

# ==============================
# 11. THỐNG KÊ SỐ LƯỢNG
# ==============================

print("\nSố lượng sinh viên mỗi cụm:")

print(result_df['Cụm'].value_counts())

# ==============================
# 12. LƯU FILE KẾT QUẢ
# ==============================

result_df.to_excel(
    'ket_qua_phan_cum.xlsx',
    index=False
)

print("\nĐã lưu file ket_qua_phan_cum.xlsx")